# 02_core_functionsCore Functions used by `03_main`.

### Read active control entries

In [0]:
def get_active_control_entries():
    query = f"""
        SELECT s_no, zip_source, zip_source_path, tgt, zip_tgt_path, unzip_tgt_path
        FROM {control_table}
        WHERE is_active = 1
    """
    return spark.sql(query).collect()

### List zip files at a source path

In [0]:
def list_zip_files(source_path):
    source_path = source_path.rstrip("/") + "/"
    return [
        f for f in dbutils.fs.ls(source_path)
        if not f.isDir() and f.name.lower().endswith(valid_extensions)
    ]

### Extract the date token from a filename 
Expected pattern: `..._<anything>_<date>.zip`, date is the second-to-last underscore-separated token, e.g. `case_data_001_20220605_20220620_20260101_235954.zip` -> `20260101`.

In [0]:
def extract_date_token(file_name):
    """
    Return the date token used for partitioning, or None if the filename
    does not have enough tokens to extract one safely.
    """
    name_without_extension = file_name.rsplit(".", 1)[0]
    parts = name_without_extension.split("_")

    if len(parts) < min_filename_tokens:
        return None

    return parts[-2]

### Copy a single file and return a log record
Tags the record with `stage="COPY"` so it can share the log table with the unzip stage.

In [0]:
def copy_zip_file(file_info, target_base_path, s_no, source):
    start_time = datetime.now()
    file_name = file_info.name
    target_base_path = target_base_path.rstrip("/") + "/"

    record = {
        "s_no": s_no,
        "stage": "COPY",
        "source": source,
        "file_name": file_name,
        "source_path": file_info.path,
        "target_path": None,
        "start_time": start_time,
    }

    date_token = extract_date_token(file_name)
    if date_token is None:
        record.update(status="SKIPPED", message="Could not extract date token from filename",
                       end_time=datetime.now())
        return record

    target_folder = f"{target_base_path}{date_token}/"
    target_file_path = f"{target_folder}{file_name}"
    record["target_path"] = target_file_path

    try:
        dbutils.fs.mkdirs(target_folder)
        dbutils.fs.cp(file_info.path, target_file_path)
        record.update(status="SUCCESS", message="Copied successfully")
    except Exception as e:
        record.update(status="FAILED", message=str(e))

    record["end_time"] = datetime.now()
    return record

### Unzip a single copied file and return a log record
- Takes a successful COPY log record (produced by `copy_zip_file`) and extracts that exact file from its copied location (`target_path`) into the matching `yyyyMMdd` folder under `unzip_base_path`. 
- Tags the record with `stage="UNZIP"`.
- Because this works off the copy record rather than re-scanning the raw folder, a run only ever unzips the files it just copied — it will not re-extract files copied by earlier runs.

In [0]:
def unzip_zip_file(copy_record, unzip_base_path):
    
    start_time = datetime.now()
    file_name = copy_record["file_name"]
    zip_path = copy_record["target_path"]
    unzip_base_path = unzip_base_path.rstrip("/") + "/"

    record = {
        "s_no": copy_record["s_no"],
        "stage": "UNZIP",
        "source": copy_record["source"],
        "file_name": file_name,
        "source_path": zip_path,
        "target_path": None,
        "start_time": start_time,
    }

    date_token = extract_date_token(file_name)
    if date_token is None:
        record.update(status="SKIPPED", message="Could not extract date token from filename",
                       end_time=datetime.now())
        return record

    target_folder = f"{unzip_base_path}{date_token}/"
    record["target_path"] = target_folder

    try:
        dbutils.fs.mkdirs(target_folder)
        local_zip_path = zip_path.replace("dbfs:", "")
        with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
            zip_ref.extractall(target_folder)
        record.update(status="SUCCESS", message="Extracted successfully")
    except zipfile.BadZipFile:
        record.update(status="FAILED", message="Invalid ZIP file")
    except Exception as e:
        record.update(status="FAILED", message=str(e))

    record["end_time"] = datetime.now()
    return record

## Write Logs

In [0]:
def write_logs(log_records, run_id):
    if not log_records:
        return

    rows = [
        (
            run_id,
            r.get("stage"),
            int(r["s_no"]) if r.get("s_no") is not None else None,
            r.get("source"),
            r.get("file_name"),
            r.get("source_path"),
            r.get("target_path"),
            r.get("status"),
            r.get("message"),
            r.get("start_time"),
            r.get("end_time"),
        )
        for r in log_records
    ]

    log_df = spark.createDataFrame(rows, schema=LOG_SCHEMA)
    log_df.write.format("delta").mode("append").saveAsTable(log_table)